In [ ]:
# ============================================
# PHASE 0: PDF Processing (Colab Version - FULLY FIXED)
# ============================================

!pip install -q pdfplumber
!pip install -q sentence-transformers
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q requests

import os
import pickle
import hashlib
import pdfplumber
import re
from typing import List, Dict, Any
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import torch

class PDFProcessor:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"📊 Colab Device: {self.device}")

        self.embedding_model = SentenceTransformer(model_name)
        if self.device == "cuda":
            self.embedding_model = self.embedding_model.to(self.device)

        # Child chunker (small chunks for precise retrieval)
        self.child_splitter = RecursiveCharacterTextSplitter(
            chunk_size=250,
            chunk_overlap=30,
            separators=["\n\n", "\n", ".", " ", ""]
        )

        # Parent chunker (larger chunks for full context)
        self.parent_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,
            chunk_overlap=100,
            separators=["\n\n", "\n", ".", " ", ""]
        )

        self.vector_db = None
        self.child_chunks = []
        self.parent_chunks = []
        self.processed_file_hashes = set()
        self.processed_chunk_hashes = set()

    def extract_text(self, pdf_path: str) -> str:
        text = ""
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text

    def extract_topic_keywords(self, text: str, top_n: int = 20) -> list:
        """Extract important keywords from PDF text to define domain"""
        from collections import Counter
        import re

        # Common stopwords to ignore
        stopwords = {'the', 'and', 'of', 'to', 'a', 'in', 'for', 'is', 'on', 'that',
                    'by', 'this', 'with', 'i', 'you', 'it', 'or', 'are', 'as', 'be',
                    'at', 'which', 'from', 'but', 'not', 'an', 'will', 'have', 'can',
                    'was', 'were', 'has', 'been', 'could', 'would', 'should', 'may'}

        # Extract words (only alphabetic, min 4 letters)
        words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())

        # Filter stopwords
        filtered_words = [w for w in words if w not in stopwords]

        # Get most common words
        word_counts = Counter(filtered_words)

        return [word for word, count in word_counts.most_common(top_n)]

    def detect_headings(self, text: str) -> List[int]:
        """Detect potential section headings for better parent chunking"""
        lines = text.split('\n')
        heading_indices = []

        for i, line in enumerate(lines):
            stripped = line.strip()
            if not stripped:
                continue
            # Heuristic: short lines, all caps, or numbered sections
            if (len(stripped) < 100 and
                (stripped.isupper() or
                 re.match(r'^(\d+\.?\s*|\•|\-|\*)[A-Z]', stripped) or
                 re.match(r'^(Chapter|Section|Part)\s+\d+', stripped, re.I))):
                heading_indices.append(i)

        return heading_indices

    def create_parent_chunks(self, text: str, pdf_name: str) -> List[Dict]:
        """Create parent chunks (full sections)"""
        parents = []

        # Try heading-based chunking first
        heading_indices = self.detect_headings(text)

        if len(heading_indices) > 1:
            # Split by headings
            lines = text.split('\n')
            for i, start_idx in enumerate(heading_indices):
                end_idx = heading_indices[i+1] if i+1 < len(heading_indices) else len(lines)
                section_text = '\n'.join(lines[start_idx:end_idx])

                parent_id = f"{pdf_name}_parent_{i}"
                parents.append({
                    "id": parent_id,
                    "text": section_text,
                    "pdf_name": pdf_name,
                    "parent_index": i,
                    "section_title": lines[start_idx].strip(),
                    "type": "parent"
                })
        else:
            # Fallback: sliding window parent chunks
            split_texts = self.parent_splitter.split_text(text)
            for i, section_text in enumerate(split_texts):
                parent_id = f"{pdf_name}_parent_{i}"
                parents.append({
                    "id": parent_id,
                    "text": section_text,
                    "pdf_name": pdf_name,
                    "parent_index": i,
                    "section_title": f"Section {i}",
                    "type": "parent"
                })

        return parents

    def create_child_chunks(self, parent_chunks: List[Dict]) -> List[Dict]:
        """Create child chunks from parent chunks"""
        children = []

        for parent in parent_chunks:
            parent_id = parent["id"]
            parent_text = parent["text"]
            section_title = parent["section_title"]

            # Split parent into smaller children
            split_texts = self.child_splitter.split_text(parent_text)

            for i, child_text in enumerate(split_texts):
                child_hash = hashlib.md5(child_text.encode()).hexdigest()

                # Skip duplicate chunks
                if child_hash in self.processed_chunk_hashes:
                    continue

                child_id = f"{parent_id}_child_{i}"
                children.append({
                    "id": child_id,
                    "text": child_text,
                    "pdf_name": parent["pdf_name"],
                    "parent_id": parent_id,
                    "parent_text": parent_text,
                    "section_title": section_title,
                    "child_index": i,
                    "text_hash": child_hash,
                    "type": "child"
                })
                self.processed_chunk_hashes.add(child_hash)

        return children

    def create_embeddings(self, chunks: List[Dict]) -> np.ndarray:
        texts = [chunk["text"] for chunk in chunks]
        embeddings = self.embedding_model.encode(
            texts,
            show_progress_bar=True,
            device=self.device
        )
        return embeddings

    def build_faiss_index(self, embeddings: np.ndarray):
        dimension = embeddings.shape[1]
        self.vector_db = faiss.IndexFlatIP(dimension)
        self.vector_db.add(embeddings.astype('float32'))

        if hasattr(faiss, 'get_num_gpus') and faiss.get_num_gpus() > 0:
            print("✅ FAISS using GPU acceleration")
        else:
            print("✅ FAISS using CPU")

    def process_pdfs(self, pdf_folder: str) -> Dict[str, Any]:
        """Main processing pipeline with parent-child chunking"""
        all_parents = []
        all_children = []

        # Track file duplicates
        for pdf_file in os.listdir(pdf_folder):
            if pdf_file.endswith('.pdf'):
                pdf_path = os.path.join(pdf_folder, pdf_file)

                # File-level deduplication
                with open(pdf_path, 'rb') as f:
                    file_hash = hashlib.md5(f.read()).hexdigest()

                if file_hash in self.processed_file_hashes:
                    print(f"⚠️ Skipping duplicate file: {pdf_file}")
                    continue

                print(f"Processing: {pdf_file}")

                # Extract text
                text = self.extract_text(pdf_path)

                # Create parent chunks (sections)
                parents = self.create_parent_chunks(text, pdf_file)

                # Create child chunks from parents
                children = self.create_child_chunks(parents)

                all_parents.extend(parents)
                all_children.extend(children)
                self.processed_file_hashes.add(file_hash)

        # Create embeddings for CHILD chunks (for retrieval)
        print("Creating embeddings for child chunks...")
        child_embeddings = self.create_embeddings(all_children)

        # Build FAISS index on child chunks
        print("Building FAISS index on child chunks...")
        self.build_faiss_index(child_embeddings)

        # Store both parent and child chunks
        self.parent_chunks = all_parents
        self.child_chunks = all_children

        return {
            "parents": all_parents,
            "children": all_children,
            "child_embeddings": child_embeddings,
            "vector_db": self.vector_db,
            "total_parents": len(all_parents),
            "total_children": len(all_children)
        }

    def get_parent_by_id(self, parent_id: str) -> Dict:
        """Retrieve parent chunk by ID"""
        for parent in self.parent_chunks:
            if parent["id"] == parent_id:
                return parent
        return None

    def save_processed_data(self, save_path: str):
        data = {
            "parents": self.parent_chunks,
            "children": self.child_chunks,
            "vector_db": self.vector_db,
            "processed_file_hashes": self.processed_file_hashes,
            "processed_chunk_hashes": self.processed_chunk_hashes
        }
        with open(save_path, 'wb') as f:
            pickle.dump(data, f)
        print(f"Saved processed data to {save_path}")

if __name__ == "__main__":
    print("=" * 50)
    print("🚀 Colab PDF Processor - Phase 0 (FIXED)")
    print("=" * 50)
    print("✅ Parent-child chunking enabled")
    print("✅ File deduplication enabled")
    print("✅ Chunk deduplication enabled")
    print("✅ FAISS IndexFlatIP active")

In [ ]:
# ============================================
# PHASE 1: Query Processing (Colab Version - FULLY FIXED)
# ============================================

import time
import threading
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
import numpy as np
import hashlib
import re
from typing import Dict, List
import faiss

class QueryRewriter:
    """Handle typos, contractions, and query normalization"""

    def __init__(self):
        # Common contractions
        self.contractions = {
            "u": "you",
            "ur": "your",
            "u r": "you are",
            "plz": "please",
            "pls": "please",
            "thx": "thanks",
            "ty": "thank you",
            "wats": "what is",
            "wat": "what",
            "wht": "what",
            "hw": "how",
            "btw": "by the way",
            "idk": "I don't know",
            "dont": "do not",
            "don't": "do not",
            "cant": "can not",
            "can't": "can not",
            "wont": "will not",
            "won't": "will not",
            "isnt": "is not",
            "isn't": "is not",
            "arent": "are not",
            "aren't": "are not"
        }

        # Technical term corrections (common typos)
        self.tech_corrections = {
            "npl": "NLP",
            "nlp": "NLP",
            "ai": "AI",
            "ml": "ML",
            "transfomer": "Transformer",
            "transfomers": "Transformers",
            "bert": "BERT",
            "gpt": "GPT",
            "lstm": "LSTM",
            "cnn": "CNN",
            "rnn": "RNN"
        }

    def normalize_query(self, query: str) -> str:
        """Clean, normalize, and fix typos in query"""
        original = query
        query = query.lower().strip()

        # Fix contractions
        words = query.split()
        for i, word in enumerate(words):
            if word in self.contractions:
                words[i] = self.contractions[word]
        query = " ".join(words)

        # Fix technical terms (preserve case)
        for typo, correction in self.tech_corrections.items():
            pattern = r'\b' + typo.lower() + r'\b'
            query = re.sub(pattern, correction, query)

        # Remove filler words (optional)
        filler_words = ["a", "an", "the", "can", "could", "would", "please", "tell", "me"]
        words = query.split()
        words = [w for w in words if w not in filler_words]
        query = " ".join(words)

        print(f"  Query rewriter: '{original}' → '{query}'")
        return query

class SemanticCache:
    """Embedding-based semantic cache using FAISS"""

    def __init__(self, embedding_model, dimension=384):
        self.embedding_model = embedding_model
        self.dimension = dimension
        self.index = faiss.IndexFlatIP(dimension)
        self.cache_data = []  # Store (query, answer, metadata)
        self.similarity_threshold = 0.99  # High threshold for cache hits

    def search(self, query_embedding: np.ndarray) -> Dict:
        """Search for similar cached query"""
        if self.index.ntotal == 0:
            return {"hit": False, "similarity": 0.0}

        try:
            distances, indices = self.index.search(
                query_embedding.reshape(1, -1).astype('float32'),
                k=1
            )

            # SAFELY get similarity
            if isinstance(distances, np.ndarray):
                if distances.ndim == 2 and distances.shape[1] > 0:
                    raw_sim = float(distances[0][0])
                elif distances.ndim == 1 and len(distances) > 0:
                    raw_sim = float(distances[0])
                elif distances.ndim == 0:
                    raw_sim = float(distances)
                else:
                    raw_sim = 0.0
            else:
                raw_sim = 0.0

            similarity = (raw_sim + 1) / 2

            if similarity >= self.similarity_threshold and indices[0][0] != -1:
                cached = self.cache_data[indices[0][0]]
                return {
                    "hit": True,
                    "similarity": similarity,
                    "original_query": cached["query"],
                    "answer": cached["answer"],
                    "metadata": cached.get("metadata", {})
                }

            return {"hit": False, "similarity": similarity}

        except Exception as e:
            print(f"⚠️ Semantic cache search error: {e}")
            return {"hit": False, "similarity": 0.0}

    def add(self, query: str, query_embedding: np.ndarray, answer: str, metadata: Dict = None):
        """Add query to cache"""
        self.index.add(query_embedding.reshape(1, -1).astype('float32'))
        self.cache_data.append({
            "query": query,
            "answer": answer,
            "metadata": metadata or {}
        })

class QueryProcessor:
    def __init__(self, pdf_processor: PDFProcessor):
        self.pdf_processor = pdf_processor

        # Initialize query rewriter
        self.query_rewriter = QueryRewriter()
        self.exact_answer_cache = {}
        # Initialize semantic cache for queries
        self.semantic_cache = SemanticCache(
            pdf_processor.embedding_model,
            dimension=384  # all-MiniLM-L6-v2 dimension
        )

        # Legacy hash cache (backward compatibility)
        self.hash_cache = {}

        # Embedding cache
        self.embedding_cache = {}

    @lru_cache(maxsize=200)
    def embed_query(self, query: str) -> np.ndarray:
        """Cache embeddings for same normalized queries"""
        embedding = self.pdf_processor.embedding_model.encode([query])[0]
        return embedding

    def normalize_ip_score(self, score: float) -> float:
        """Convert IP score to 0-1 range"""
        return max(0.0, (score + 1) / 2)

    def faiss_search_parallel(self, query_embedding: np.ndarray, top_k: int = 20):
        """FAISS search in parallel thread - ULTIMATE FIX"""
        try:
            distances, indices = self.pdf_processor.vector_db.search(
                query_embedding.reshape(1, -1).astype('float32'),
                top_k
            )

            # CASE 1: FAISS returned scalars (top_k=1)
            if isinstance(distances, (float, int, np.float32, np.float64)) or np.isscalar(distances):
                return np.array([float(distances)]), np.array([int(indices)])

            # CASE 2: FAISS returned 2D arrays (normal case)
            if isinstance(distances, np.ndarray):
                if distances.ndim == 2:
                    return distances[0], indices[0]
                elif distances.ndim == 1:
                    return distances, indices
                elif distances.ndim == 0:
                    return np.array([float(distances)]), np.array([int(indices)])

            # CASE 3: FAISS returned something else
            try:
                # Try to access as sequence
                if len(distances) > 0:
                    if isinstance(distances[0], (list, np.ndarray)):
                        return np.array(distances[0]), np.array(indices[0])
                    else:
                        return np.array([float(distances[0])]), np.array([int(indices[0])])
            except:
                pass

            # Ultimate fallback
            return np.array([0.0]), np.array([0])

        except Exception as e:
            print(f"⚠️ FAISS search error: {e}")
            return np.array([0.0]), np.array([0])

    def keyword_search_parallel(self, query: str, top_k: int = 10):
        """Simple keyword matching"""
        query_words = set(query.lower().split())
        scores = []

        # Search against child chunks
        for i, chunk in enumerate(self.pdf_processor.child_chunks):
            chunk_words = set(chunk["text"].lower().split())
            common_words = len(query_words.intersection(chunk_words))
            score = common_words / max(len(query_words), 1)
            scores.append((score, i))

        scores.sort(reverse=True, key=lambda x: x[0])
        return [idx for _, idx in scores[:top_k]]

    def process_query_parallel(self, query: str) -> Dict:
        """Process query with parallel operations - FIXED"""
        if query in self.exact_answer_cache:
            print("  ✅ EXACT CACHE HIT!")
            return {
                "query": query,
                "rewritten_query": query,
                "retrieved_chunks": [],
                "max_similarity": 1.0,
                "cache_hit": True,
                "semantic_cache": False,
                "cached_answer": self.exact_answer_cache[query]
            }
        # STEP 1: Rewrite query (fix typos, normalize)
        rewritten_query = self.query_rewriter.normalize_query(query)
        print(f"Original: '{query}' → Rewritten: '{rewritten_query}'")
        # STEP 2: Check semantic cache with rewritten query
        query_embedding = self.embed_query(rewritten_query)
        cache_result = self.semantic_cache.search(query_embedding)

        if cache_result and cache_result.get("hit"):
            # Cache hit! Return cached answer
            return {
                "query": query,
                "rewritten_query": rewritten_query,
                "retrieved_chunks": [],
                "max_similarity": cache_result["similarity"],
                "faiss_indices": [],
                "keyword_indices": [],
                "distances": [],
                "cache_hit": True,
                "cached_answer": cache_result["answer"],
                "cached_query": cache_result["original_query"]
            }

        # STEP 3: Cache miss - process normally
        print("  Semantic cache MISS - processing...")

        max_workers = 4

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            embedding_future = executor.submit(self.embed_query, rewritten_query)

            def faiss_task():
                embedding = embedding_future.result()
                return self.faiss_search_parallel(embedding, top_k=20)
            faiss_future = executor.submit(faiss_task)

            keyword_future = executor.submit(self.keyword_search_parallel, rewritten_query)

        # Get results
        distances, faiss_indices = faiss_future.result()
        keyword_indices = keyword_future.result()

        # ===== SAFE DISTANCE HANDLING =====
        # Convert distances to 1D array safely
        if isinstance(distances, np.ndarray):
            if distances.ndim == 0:
                dist_array = np.array([float(distances)])
            elif distances.ndim == 1:
                dist_array = distances
            elif distances.ndim == 2:
                dist_array = distances[0]
            else:
                dist_array = np.array([0.0])
        else:
            try:
                dist_array = np.array([float(d) for d in distances])
            except:
                dist_array = np.array([0.0])

        # Get max similarity
        if len(dist_array) > 0:
            max_similarity = self.normalize_ip_score(float(dist_array[0]))
        else:
            max_similarity = 0.0

        # ===== SAFE INDICES HANDLING =====
        # Convert faiss_indices to list of ints
        if isinstance(faiss_indices, np.ndarray):
            if faiss_indices.ndim == 0:
                faiss_idx_list = [int(faiss_indices)]
            elif faiss_indices.ndim == 1:
                faiss_idx_list = [int(idx) for idx in faiss_indices]
            elif faiss_indices.ndim == 2:
                faiss_idx_list = [int(idx) for idx in faiss_indices[0]]
            else:
                faiss_idx_list = []
        else:
            try:
                faiss_idx_list = [int(idx) for idx in faiss_indices]
            except:
                try:
                    faiss_idx_list = [int(faiss_indices)]
                except:
                    faiss_idx_list = []

        # Ensure we have valid indices
        faiss_idx_list = [idx for idx in faiss_idx_list if idx >= 0 and idx < len(self.pdf_processor.child_chunks)]

        # Combine results
        all_indices = list(set(list(faiss_idx_list) + keyword_indices))
        retrieved_child_chunks = []

        for i in all_indices:
            if i < len(self.pdf_processor.child_chunks):
                chunk = self.pdf_processor.child_chunks[i].copy()

                # Calculate similarity
                if i in faiss_idx_list:
                    try:
                        idx_pos = list(faiss_idx_list).index(i)
                        if idx_pos < len(dist_array):
                            similarity = self.normalize_ip_score(float(dist_array[idx_pos]))
                        else:
                            similarity = 0.2
                    except:
                        similarity = 0.2
                else:
                    similarity = 0.2

                chunk["similarity"] = similarity
                retrieved_child_chunks.append(chunk)

        # Sort by similarity
        retrieved_child_chunks.sort(key=lambda x: x.get("similarity", 0), reverse=True)

                # Handle both scalar and array cases
        if isinstance(distances[0], np.ndarray):
            max_similarity = self.normalize_ip_score(distances[0][0]) if len(distances[0]) > 0 else 0.0
        else:
            max_similarity = self.normalize_ip_score(distances[0])  # scalar case

        result = {
            "query": query,
            "rewritten_query": rewritten_query,
            "retrieved_chunks": retrieved_child_chunks,  # These are CHILDREN
            "max_similarity": max_similarity,
            "faiss_indices": faiss_indices,
            "keyword_indices": keyword_indices,
            "distances": distances,
            "cache_hit": False
        }

        return result

if __name__ == "__main__":
    print("✅ Phase 1: Query Processor FULLY FIXED")
    print("   - Query rewriter with typo correction")
    print("   - Embedding-based semantic cache (FAISS)")
    print("   - Normalized IP scores")
    print("   - Child chunk retrieval")

In [ ]:
# ============================================
# PHASE 2: Smart Routing (Colab Version - FIXED)
# ============================================

from typing import Dict
import random

class SmartRouter:
    def __init__(self):
        # NEW: Three-tier thresholds
        self.unrelated_threshold = 0.35      # Below this = completely unrelated
        self.external_fallback_threshold = 0.55  # 0.25-0.55 = try external
        self.high_confidence_threshold = 0.70     # Above this = use documents only
        self.specific_query_length = 8
        self.soft_refusal = True

    def route_query(self, query: str, max_similarity: float, pdf_has_content: bool = True) -> Dict:
        """Make routing decisions based on query and similarity"""
        query_length = len(query.split())

        # Decision 1: No PDFs uploaded
        if not pdf_has_content:
            return {
                "route": "external_only",
                "action": "external_fallback",
                "reason": "No documents uploaded",
                "confidence": 0.0
            }

        # Decision 2: Completely unrelated
        if max_similarity < self.unrelated_threshold:
            return {
                "route": "unrelated",
                "action": "refuse",
                "reason": f"Query unrelated to documents (similarity: {max_similarity:.3f})",
                "confidence": float(max_similarity)
            }

        # Decision 3: External fallback needed (topic related but insufficient)
        elif max_similarity < self.external_fallback_threshold:
            return {
                "route": "external_fallback",
                "action": "external_search",
                "reason": f"Low confidence but topic related (similarity: {max_similarity:.3f})",
                "confidence": float(max_similarity)
            }

        # Decision 4: High confidence - documents only
        elif max_similarity > self.high_confidence_threshold:
            return {
                "route": "high_confidence",
                "action": "direct_answer",
                "reason": f"High confidence match (similarity: {max_similarity:.3f})",
                "confidence": float(max_similarity)
            }

        # Decision 5: Specific query
        elif query_length > self.specific_query_length:
            return {
                "route": "specific_query",
                "action": "direct_retrieval",
                "reason": f"Specific query ({query_length} words)",
                "confidence": float(max_similarity)
            }

        # Decision 6: Needs processing
        else:
            return {
                "route": "needs_processing",
                "action": "adaptive_retrieval",
                "reason": f"Medium confidence (similarity: {max_similarity:.3f})",
                "confidence": float(max_similarity)
            }

In [ ]:
# ============================================
# PHASE 3: Adaptive Processing (Colab Version - FIXED)
# ============================================

!pip install -q rake-nltk
!pip install -q transformers

from rake_nltk import Rake
from transformers import pipeline
from typing import List, Dict
import numpy as np

class AdaptiveProcessor:
    def __init__(self, pdf_processor: PDFProcessor):
        self.pdf_processor = pdf_processor
        self.rake = Rake()

        try:
            from sentence_transformers import CrossEncoder
            self.sufficiency_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
            print("✅ Cross-Encoder loaded")
        except Exception as e:
            print(f"⚠️ Cross-Encoder not available: {e}")
            self.sufficiency_model = None

        try:
            self.expansion_model = pipeline(
                "text-generation",
                model="google/flan-t5-small",
                device=0 if self.pdf_processor.device == "cuda" else -1,
                max_length=50
            )
            print("✅ FLAN-T5 loaded")
        except Exception as e:
            print(f"⚠️ FLAN-T5 not available: {e}")
            self.expansion_model = None

        # Relevance threshold for safe expansion
        self.relevance_threshold = 0.6  # Only expand if chunks are relevant

    def check_relevance(self, query: str, chunks: List[Dict]) -> float:
        """Check if retrieved chunks are actually relevant to query"""
        if not chunks:
            return 0.0

        # Use top chunk's similarity score
        top_similarity = chunks[0].get("similarity", 0)
        return top_similarity

    def needs_expansion(self, query: str, retrieved_chunks: List) -> bool:
        """Check if query needs expansion"""
        if len(query.split()) < 3:
            return True
        if len(retrieved_chunks) < 3:
            return True

        vague_indicators = ["what", "how", "why", "explain", "tell me about"]
        if any(indicator in query.lower() for indicator in vague_indicators):
            return True

        return False

    def expand_query_simple(self, query: str, retrieved_chunks: List) -> str:
        """Expand query using RAKE - but only if chunks are relevant"""

        # Check relevance first
        relevance = self.check_relevance(query, retrieved_chunks)

        if relevance < self.relevance_threshold:
            # Chunks are not relevant - use synonym expansion instead
            print(f"  ⚠️ Low relevance ({relevance:.2f}) - using synonym expansion")
            synonyms = {
                "explain": "definition overview",
                "what": "definition",
                "how": "process method",
                "why": "reason cause"
            }
            words = query.lower().split()
            for i, word in enumerate(words):
                if word in synonyms:
                    words[i] = synonyms[word]
            return " ".join(words)

        # Chunks are relevant - safe to expand with keywords
        combined_text = " ".join([chunk["text"] for chunk in retrieved_chunks[:3]])
        self.rake.extract_keywords_from_text(combined_text)
        keywords = self.rake.get_ranked_phrases()[:3]

        expanded_query = query
        for keyword in keywords:
            if keyword not in expanded_query.lower():
                expanded_query += " " + keyword

        return expanded_query.strip()

    def expand_query_llm(self, query: str, retrieved_chunks: List) -> str:
        """Expand query using FLAN-T5 - with relevance check"""
        if not self.expansion_model:
            return self.expand_query_simple(query, retrieved_chunks)

        # Check relevance first
        relevance = self.check_relevance(query, retrieved_chunks)

        if relevance < self.relevance_threshold:
            # Skip LLM expansion for irrelevant chunks
            return self.expand_query_simple(query, retrieved_chunks)

        context = " ".join([chunk["text"][:200] for chunk in retrieved_chunks[:2]])

        prompt = f"""
        Query: {query}
        Context: {context}
        Suggest 2-3 keywords from the context that would help find more relevant information.
        Keywords:"""

        try:
            expanded_result = self.expansion_model(
                prompt,
                max_length=50,
                do_sample=False,
                num_return_sequences=1
            )

            expanded = expanded_result[0]['generated_text']

            if "Keywords:" in expanded:
                keywords_part = expanded.split("Keywords:")[-1].strip()
            else:
                keywords_part = expanded.strip()

            keywords = [kw.strip() for kw in keywords_part.replace('\n', ',').split(',') if kw.strip()]
            expanded_query = query + " " + " ".join(keywords[:3])
            return expanded_query
        except Exception as e:
            print(f"⚠️ Query expansion failed: {e}")
            return self.expand_query_simple(query, retrieved_chunks)

    def check_sufficiency(self, query: str, chunks: List[Dict]) -> bool:
        """Check if chunks are sufficient to answer query"""
        if not self.sufficiency_model:
            total_text = " ".join([chunk["text"] for chunk in chunks[:3]])
            query_words = set(query.lower().split())
            text_words = set(total_text.lower().split())
            overlap = len(query_words.intersection(text_words))
            return overlap >= len(query_words) * 0.3

        pairs = [(query, chunk["text"]) for chunk in chunks[:5]]
        scores = self.sufficiency_model.predict(pairs)
        avg_score = np.mean(scores)

        # Loosened threshold
        return avg_score > 2.0

    def hybrid_search(self, query: str, original_chunks: List, max_loops: int = 3):
        """Perform hybrid search with safe expansion"""
        current_query = query
        all_chunks = original_chunks.copy()

        for attempt in range(max_loops):
            if self.check_sufficiency(current_query, all_chunks):
                return all_chunks, attempt + 1

            if self.needs_expansion(current_query, all_chunks):
                current_query = self.expand_query_llm(query, all_chunks)
                print(f"  Expanded query (attempt {attempt+1}): {current_query}")

            # Search again with current query
            # Search again with current query
            embedding = self.pdf_processor.embedding_model.encode([current_query])[0]
            distances, indices = self.pdf_processor.vector_db.search(
                embedding.reshape(1, -1).astype('float32'),
                k=15
            )

            # ===== SAFE DISTANCE HANDLING =====
            if isinstance(distances, np.ndarray):
                if distances.ndim == 0:
                    dist_array = np.array([float(distances)])
                    idx_array = np.array([int(indices)])
                elif distances.ndim == 1:
                    dist_array = distances
                    idx_array = indices if isinstance(indices, np.ndarray) else np.array([indices])
                elif distances.ndim == 2:
                    dist_array = distances[0]
                    idx_array = indices[0]
                else:
                    dist_array = np.array([])
                    idx_array = np.array([])
            else:
                try:
                    dist_array = np.array([float(d) for d in distances])
                    idx_array = np.array([int(i) for i in indices])
                except:
                    dist_array = np.array([])
                    idx_array = np.array([])

            # Add new chunks (deduplicate)
            for i, idx in enumerate(idx_array):
                if idx not in [c.get("index", -1) for c in all_chunks]:
                    if 0 <= idx < len(self.pdf_processor.child_chunks):
                        chunk = self.pdf_processor.child_chunks[idx].copy()
                        if i < len(dist_array):
                            similarity = (float(dist_array[i]) + 1) / 2
                            chunk["similarity"] = similarity
                        else:
                            chunk["similarity"] = 0.3
                        chunk["index"] = idx
                        all_chunks.append(chunk)

            all_chunks = sorted(all_chunks, key=lambda x: x.get("similarity", 0), reverse=True)[:20]

        return all_chunks, max_loops

if __name__ == "__main__":
    print("✅ Phase 3: Adaptive Processor FIXED")
    print("   - Relevance check BEFORE expansion")
    print("   - Synonym fallback for irrelevant chunks")
    print("   - Loosened sufficiency threshold")
    print("   - Increased retrieval k to 15")

In [ ]:
# CELL: Load Colab Secrets (add this BEFORE AnswerGenerator cell)
from google.colab import userdata
import os

try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["OPENAI_BASE_URL"] = userdata.get('OPENAI_BASE_URL')
    os.environ["OPENAI_MODEL"] = userdata.get('OPENAI_MODEL')
    print("✅ Colab Secrets loaded successfully!")
except Exception as e:
    print(f"⚠️ Could not load Colab Secrets: {e}")
    print("Please set environment variables manually")

In [ ]:
# ============================================
# PHASE 3.5: External Knowledge Retriever
# ============================================

import requests
import json
from typing import List, Dict
import time

class ExternalRetriever:
    def __init__(self):
        self.sources = {
            "wikipedia": {
                "api_url": "https://en.wikipedia.org/api/rest_v1/page/summary/",
                "search_url": "https://en.wikipedia.org/w/api.php",
                "enabled": True
            },
            "arxiv": {
                "api_url": "http://export.arxiv.org/api/query",
                "enabled": True
            }
        }

    def search_wikipedia(self, query: str, max_results: int = 2) -> List[Dict]:
        """Search Wikipedia and return summaries"""
        results = []
        headers = {
        'User-Agent': 'AcademicRAGBot/1.0 (showbiya.se23@bitsathy.ac.in) - Academic RAG Assistant'
    }
        try:
            # Search for pages
            search_params = {
                "action": "query",
                "list": "search",
                "srsearch": query,
                "format": "json",
                "srlimit": max_results
            }

            response = requests.get(
                self.sources["wikipedia"]["search_url"],
                params=search_params,
                headers=headers,
                timeout=10
            )

            if response.status_code != 200:
                return []

            data = response.json()
            search_results = data.get("query", {}).get("search", [])

            for item in search_results:
                page_title = item.get("title")

                # Get page summary
                summary_response = requests.get(
                    f"{self.sources['wikipedia']['api_url']}{page_title.replace(' ', '_')}",
                    timeout=10
                )

                if summary_response.status_code == 200:
                    summary_data = summary_response.json()

                    results.append({
                        "text": summary_data.get("extract", "No summary available"),
                        "title": page_title,
                        "source": "wikipedia",
                        "url": f"https://en.wikipedia.org/wiki/{page_title.replace(' ', '_')}",
                        "relevance_score": 0.0  # Will calculate
                    })

            time.sleep(0.1)  # Be nice to the API

        except Exception as e:
            print(f"⚠️ Wikipedia search error: {e}")

        return results

    def search_arxiv(self, query: str, max_results: int = 1) -> List[Dict]:
        """Search arXiv for academic papers"""
        results = []

        try:
            params = {
                "search_query": f"all:{query}",
                "start": 0,
                "max_results": max_results,
                "sortBy": "relevance",
                "sortOrder": "descending"
            }

            response = requests.get(
                self.sources["arxiv"]["api_url"],
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                return []

            # Parse XML response (simplified)
            import xml.etree.ElementTree as ET
            root = ET.fromstring(response.content)

            namespace = {"atom": "http://www.w3.org/2005/Atom"}
            entries = root.findall("atom:entry", namespace)

            for entry in entries[:max_results]:
                title = entry.find("atom:title", namespace)
                summary = entry.find("atom:summary", namespace)
                link = entry.find("atom:link", namespace)

                link_url = link.get("href") if link is not None else ""

                results.append({
                    "text": f"Title: {title.text if title is not None else 'Unknown'}\n\nAbstract: {summary.text if summary is not None else 'No abstract'}",
                    "title": title.text if title is not None else "Unknown",
                    "source": "arxiv",
                    "url": link_url,
                    "relevance_score": 0.0
                })

        except Exception as e:
            print(f"⚠️ arXiv search error: {e}")

        return results

    def calculate_relevance(self, query: str, text: str) -> float:
        """Simple keyword overlap relevance score"""
        query_words = set(query.lower().split())
        text_words = set(text.lower().split()[:500])  # Limit text size

        if not query_words:
            return 0.0

        overlap = len(query_words.intersection(text_words))
        return min(overlap / len(query_words), 1.0)

    def search(self, query: str, min_relevance: float = 0.3) -> List[Dict]:
        """Search all enabled external sources"""
        all_results = []

        # Search Wikipedia
        if self.sources["wikipedia"]["enabled"]:
            wiki_results = self.search_wikipedia(query, max_results=2)
            for result in wiki_results:
                result["relevance_score"] = self.calculate_relevance(query, result["text"])
            all_results.extend(wiki_results)

        # Search arXiv
        if self.sources["arxiv"]["enabled"]:
            arxiv_results = self.search_arxiv(query, max_results=1)
            for result in arxiv_results:
                result["relevance_score"] = self.calculate_relevance(query, result["text"])
            all_results.extend(arxiv_results)

        # Filter by relevance
        filtered = [r for r in all_results if r["relevance_score"] >= min_relevance]

        # Sort by relevance
        filtered.sort(key=lambda x: x["relevance_score"], reverse=True)

        return filtered[:3]  # Max 3 external chunks

In [ ]:
# ============================================
# PHASE 4: Answer Generation (Colab Version - FULLY FIXED)
# ============================================

!pip install -q openai

import openai
from openai import OpenAI
from datetime import datetime
from typing import List, Dict
import os
import hashlib
import numpy as np

class MMR:
    """Maximum Marginal Relevance for diverse chunk selection"""

    def __init__(self, lambda_param: float = 0.7):
        self.lambda_param = lambda_param  # Balance relevance (1) vs diversity (0)

    def compute_similarity(self, emb1: np.ndarray, emb2: np.ndarray) -> float:
        """Cosine similarity between two embeddings"""
        return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))

    def select_diverse_chunks(self, chunks: List[Dict], query_embedding: np.ndarray,
                             chunk_embeddings: List[np.ndarray], top_k: int = 5) -> List[Dict]:
        """Select diverse chunks using MMR"""
        if len(chunks) <= top_k:
            return chunks

        selected = []
        remaining = list(range(len(chunks)))

        # First: select highest relevance chunk
        relevance_scores = [chunk.get("similarity", 0) for chunk in chunks]
        first_idx = np.argmax(relevance_scores)
        selected.append(first_idx)
        remaining.remove(first_idx)

        # Then: select chunks balancing relevance and diversity
        while len(selected) < top_k and remaining:
            mmr_scores = []

            for i in remaining:
                # Relevance score
                relevance = chunks[i].get("similarity", 0)

                # Diversity score (max similarity to already selected)
                max_sim = 0
                for j in selected:
                    sim = self.compute_similarity(
                        chunk_embeddings[i],
                        chunk_embeddings[j]
                    )
                    max_sim = max(max_sim, sim)

                # MMR score
                mmr = self.lambda_param * relevance - (1 - self.lambda_param) * max_sim
                mmr_scores.append(mmr)

            # Select chunk with highest MMR
            best_idx = remaining[np.argmax(mmr_scores)]
            selected.append(best_idx)
            remaining.remove(best_idx)

        return [chunks[i] for i in selected]

class SemanticAnswerCache:
    """Embedding-based answer cache using FAISS"""

    def __init__(self, embedding_model, dimension=384):
        self.embedding_model = embedding_model
        self.dimension = dimension
        self.index = faiss.IndexFlatIP(dimension)
        self.cache_data = []  # Store (query, answer, metadata)
        self.similarity_threshold = 0.92

    def search(self, query_embedding: np.ndarray) -> Dict:
        if self.index.ntotal == 0:
            return None

        distances, indices = self.index.search(
            query_embedding.reshape(1, -1).astype('float32'),
            k=1
        )

        similarity = (distances[0][0] + 1) / 2 if len(distances[0]) > 0 else 0

        if similarity >= self.similarity_threshold and indices[0][0] != -1:
            cached = self.cache_data[indices[0][0]]
            return {
                "hit": True,
                "similarity": similarity,
                "answer": cached["answer"],
                "metadata": cached.get("metadata", {})
            }

        return {"hit": False, "similarity": similarity}

    def add(self, query: str, query_embedding: np.ndarray, answer: str, metadata: Dict = None):
        self.index.add(query_embedding.reshape(1, -1).astype('float32'))
        self.cache_data.append({
            "query": query,
            "answer": answer,
            "metadata": metadata or {}
        })

class AnswerGenerator:
    def __init__(self, api_key: str = None, base_url: str = None, model: str = None):
        self.api_key = api_key or os.getenv("OPENAI_API_KEY")
        self.base_url = base_url or os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
        self.model = model or os.getenv("OPENAI_MODEL", "gpt-4.1-nano")

        # Legacy hash cache
        self.hash_cache = {}

        # New semantic answer cache
        self.semantic_cache = None  # Will initialize later with embedding model

        self.client = None
        if self.api_key:
            self.client = OpenAI(
                api_key=self.api_key,
                base_url=self.base_url,
            )
            print(f"✅ OpenAI client initialized with model: {self.model}")

        # MMR for diversity
        self.mmr = MMR(lambda_param=0.7)

        # Temperature set to 0 for deterministic answers
        self.temperature = 0.0
        self.type_aware_cache = {}  # For query-type specific caching

    def set_embedding_model(self, embedding_model):
        """Set embedding model for semantic cache"""
        self.semantic_cache = SemanticAnswerCache(embedding_model)

    def get_parent_chunks(self, child_chunks: List[Dict], pdf_processor: PDFProcessor) -> List[Dict]:
        """Convert child chunks to parent chunks (deduplicated)"""
        parent_map = {}

        for child in child_chunks:
            parent_id = child.get("parent_id")
            if parent_id and parent_id not in parent_map:
                parent = pdf_processor.get_parent_by_id(parent_id)
                if parent:
                    # Add relevance score from child
                    parent_copy = parent.copy()
                    parent_copy["similarity"] = child.get("similarity", 0)
                    parent_copy["child_count"] = 1
                    parent_map[parent_id] = parent_copy
            elif parent_id in parent_map:
                # Update similarity (take max)
                parent_map[parent_id]["similarity"] = max(
                    parent_map[parent_id]["similarity"],
                    child.get("similarity", 0)
                )
                parent_map[parent_id]["child_count"] += 1

        # Convert back to list and sort by similarity
        parents = list(parent_map.values())
        parents.sort(key=lambda x: x.get("similarity", 0), reverse=True)

        print(f"  Converted {len(child_chunks)} children → {len(parents)} unique parents")
        return parents

    def format_context(self, parent_chunks: List[Dict], max_chunks: int = 5) -> str:
        """Format parent chunks for LLM context"""
        context_parts = []

        for i, chunk in enumerate(parent_chunks[:max_chunks]):
            source = chunk.get("pdf_name", "Document")
            section = chunk.get("section_title", f"Section {i}")

            context_parts.append(
                f"[Source: {source}, {section}]\n"
                f"{chunk['text']}\n"
            )

        return "\n".join(context_parts)


    def detect_query_type(self, query: str) -> str:
        """Detect if query asks for list, explanation, definition, etc."""
        query_lower = query.lower()

        if any(word in query_lower for word in ["simple terms", "simply", "easy", "layman"]):
            return "simple_explanation"
        elif any(word in query_lower for word in ["list", "what are", "types of", "kinds of", "categories", "examples of"]):
            return "list"
        elif any(word in query_lower for word in ["explain", "describe", "how does", "how do"]):
            return "explain"
        elif any(word in query_lower for word in ["define", "what is", "definition"]):
            return "definition"
        elif any(word in query_lower for word in ["compare", "difference", "versus", "vs"]):
            return "compare"
        else:
            return "general"

    def build_prompt(self, query: str, context: str, query_type: str, has_external: bool = False) -> str:
        """Build prompt with source attribution for external content"""

        source_instruction = ""
        if has_external:
            source_instruction = """
    IMPORTANT - SOURCE PRIORITIZATION:
    1. Prioritize information from user's uploaded documents over external sources
    2. When using external sources (Wikipedia, arXiv), clearly state "Based on Wikipedia" or "Based on arXiv"
    3. If documents and external sources conflict, trust the documents
    4. Always include URLs for external sources in your citations
    """

        base_instructions = f"""
    1. Base your answer on the provided documents
    2. If the answer isn't in the documents but external sources are provided, use them
    3. If the answer isn't in any provided source, say so
    4. Include citations like [Source: X, Section Y] or [Wikipedia: URL]
    {source_instruction}
    """

        # Rest of your type_instructions remains the same
        type_instructions = {
            "list": """
    5. Format your answer as a COMPLETE bulleted list
    6. Include ALL items mentioned
    """,
            "explain": """
    5. Provide a thorough explanation
    6. Break down complex concepts into steps
    """,
            "definition": """
    5. Provide a clear, concise definition
    6. Include key characteristics
    """,
            "compare": """
    5. Present comparison in structured format
    6. List similarities first, then differences
    """,
            "general": """
    5. Answer directly and concisely
    6. Use bullet points for multiple points
    """
        }

        prompt = f"""Based on the following information, answer the question.

    Documents:
    {context}

    Question: {query}

    Instructions:{base_instructions}{type_instructions.get(query_type, "")}

    Answer:"""

        return prompt

    def generate_with_gpt(self, query: str, context: str, query_type: str = "general", has_external: bool = False) -> Dict:
        """Generate answer with source awareness"""
        if not self.client:
            return {
                "answer": "OpenAI API client not configured",
                "model": "none",
                "tokens": 0,
                "error": "No API client"
            }

        prompt = self.build_prompt(query, context, query_type, has_external)

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a helpful academic assistant. Provide complete, accurate information from the documents."},
                    {"role": "user", "content": prompt}
                ],
                temperature=self.temperature,  # Now 0.0 for deterministic answers
                max_tokens=800  # Increased for longer responses
            )

            answer = response.choices[0].message.content

            return {
                "answer": answer,
                "model": self.model,
                "tokens": response.usage.total_tokens,
                "timestamp": datetime.now().isoformat()
            }

        except Exception as e:
            return {
                "answer": f"Error generating answer: {str(e)}",
                "model": self.model,
                "tokens": 0,
                "error": str(e)
            }

    def get_answer(self, query: str, child_chunks: List[Dict],
               pdf_processor: PDFProcessor, query_embedding: np.ndarray = None,
               skip_cache: bool = False) -> Dict:
        """Main answer generation with parent retrieval, MMR, and semantic caching"""

        # STEP 1: Convert child chunks → parent chunks (deduplicated)
        parent_chunks = self.get_parent_chunks(child_chunks, pdf_processor)

        if not parent_chunks:
            return {
                "answer": "No relevant documents found.",
                "chunks_used": 0,
                "cache_hit": False
            }

        # STEP 2: Apply MMR for diversity (if we have embeddings)
        if query_embedding is not None and len(parent_chunks) > 3:
            # Get embeddings for parent chunks
            parent_texts = [chunk["text"][:500] for chunk in parent_chunks[:10]]
            parent_embeddings = pdf_processor.embedding_model.encode(parent_texts)

            # Select diverse parents
            parent_chunks = self.mmr.select_diverse_chunks(
                parent_chunks[:10],
                query_embedding,
                parent_embeddings,
                top_k=5
            )
            print(f"  MMR applied: selected {len(parent_chunks)} diverse parents")

        # STEP 3: Detect query type for cache differentiation
        query_type = self.detect_query_type(query)
        print(f"  Query type detected: {query_type}")

        # STEP 4: Check semantic answer cache WITH QUERY TYPE
        if self.semantic_cache and query_embedding is not None:
            # Create query type aware cache key
            cache_key = hashlib.md5(
                query_embedding.tobytes() + query_type.encode()
            ).hexdigest()

            # Check if this specific query+type is in cache
            if hasattr(self, 'type_aware_cache') and cache_key in self.type_aware_cache:
                print(f"  ✅ Answer cache HIT! (type: {query_type})")
                return self.type_aware_cache[cache_key]

            # Fallback to regular semantic cache (but check if types match)
            cache_result = self.semantic_cache.search(query_embedding)
            if cache_result and cache_result.get("hit"):
                cached_metadata = cache_result.get("metadata", {})
                cached_type = cached_metadata.get("query_type", "unknown")

                # Only return cache hit if query types match
                if cached_type == query_type:
                    print(f"  ✅ Answer cache HIT! (semantic, type match: {query_type})")
                    result = {
                        "answer": cache_result["answer"],
                        "model": "cached",
                        "tokens": 0,
                        "cache_hit": True,
                        "semantic_cache": True,
                        "similarity": cache_result["similarity"],
                        "chunks_used": len(parent_chunks[:5]),
                        "query_type": query_type
                    }
                    return result
                else:
                    print(f"  ⚠️ Cache miss: type mismatch (cached: {cached_type}, current: {query_type})")

        print("  Answer cache MISS - generating...")

        # STEP 4: Detect query type for structured output
        query_type = self.detect_query_type(query)
        print(f"  Detected query type: {query_type}")

        # STEP 5: Dynamic context size based on query type
        if query_type == "list":
            max_chunks = 8  # Need more chunks for complete lists
        elif query_type == "explain":
            max_chunks = 6
        else:
            max_chunks = 5

        context = self.format_context(parent_chunks, max_chunks=max_chunks)

        # STEP 6: Generate answer
        result = self.generate_with_gpt(query, context, query_type)
        result["cache_hit"] = False
        result["semantic_cache"] = False
        result["chunks_used"] = len(parent_chunks[:max_chunks])
        result["parents_used"] = len(parent_chunks[:max_chunks])
        result["query_type"] = query_type


        # STEP 7: Store in semantic cache with query type
        if self.semantic_cache and query_embedding is not None and not skip_cache:

            # Check if answer is a refusal message (should not be cached)
            refusal_indicators = [
                "do not contain",
                "cannot answer",
                "unrelated",
                "not found",
                "no information",
                "does not mention",
                "does not contain",
                "not covered",        # ADD THIS
                "no specific information",  # ADD THIS
                "not available"
            ]

            is_refusal = any(indicator in result["answer"].lower() for indicator in refusal_indicators)

            # Only cache non-refusal answers with confidence > 0.7
            if not is_refusal:
                # Store in semantic cache with metadata
                self.semantic_cache.add(
                    query,
                    query_embedding,
                    result["answer"],
                    metadata={"query_type": query_type}
                )

                # Also store in type-aware cache
                cache_key = hashlib.md5(
                    query_embedding.tobytes() + query_type.encode()
                ).hexdigest()

                # Create a copy with cache_hit = True for future retrievals
                cached_result = result.copy()
                cached_result["cache_hit"] = True
                cached_result["semantic_cache"] = True
                self.type_aware_cache[cache_key] = cached_result

                print(f"  Stored answer in cache (confidence: {result.get('confidence', 0):.2f})")
            else:
                print(f"  Not caching answer (refusal: {is_refusal}, confidence: {result.get('confidence', 0):.2f})")
        else:
            if skip_cache:
                print(f"  Not caching answer (skip_cache=True)")

        return result

if __name__ == "__main__":
    print("✅ Phase 4: Answer Generator FULLY FIXED")
    print("   - Parent retrieval (child → parent)")
    print("   - Parent deduplication")
    print("   - MMR diversity reranking")
    print("   - Dynamic context size")
    print("   - Structured output prompts")
    print("   - Temperature = 0 (deterministic)")
    print("   - Semantic answer cache")

In [ ]:
# ============================================
# PHASE 5: Answer Validation (Colab Version - FIXED)
# ============================================

from transformers import pipeline
from typing import List, Dict, Any
import numpy as np

class AnswerValidator:
    def __init__(self):
        try:
            self.nli_model = pipeline(
                "text-classification",
                model="facebook/bart-large-mnli",
                device=0
            )
            print("✅ BART-MNLI loaded (GPU)")
        except Exception as e:
            print(f"⚠️ BART-MNLI not available: {e}")
            self.nli_model = None

    def heuristic_correctness(self, answer: str, query: str) -> float:
        """Simple heuristic check for answer correctness"""
        score = 0.0

        if len(answer.split()) > 10:
            score += 0.2

        query_words = set(query.lower().split())
        answer_words = set(answer.lower().split())
        overlap = len(query_words.intersection(answer_words))
        if overlap > 0:
            score += min(0.3, overlap * 0.1)

        if "[" in answer and "]" in answer:
            score += 0.2

        uncertain_phrases = [
            "i don't know", "not in the documents",
            "cannot answer", "no information"
        ]
        if not any(phrase in answer.lower() for phrase in uncertain_phrases):
            score += 0.3

        return min(float(score), 1.0)

    def hallucination_check(self, answer: str, context: str) -> Dict:
        """Check if answer is hallucinated using NLI - FIXED VERSION"""
        if not self.nli_model:
            return {
                "is_hallucinated": False,
                "confidence": 0.5,
                "reason": "NLI model not available"
            }

        premise = context[:1000]
        hypothesis = answer[:500]

        try:
            # CORRECT WAY: Pass (premise, hypothesis) as separate arguments
            # Or use the pipeline with a single string containing both
            result = self.nli_model(f"{premise} [SEP] {hypothesis}")

            # Extract entailment score
            entailment_score = 0.0
            if isinstance(result, list) and len(result) > 0:
                # For sequence classification output
                if result[0].get('label') == 'ENTAILMENT':
                    entailment_score = result[0].get('score', 0.0)
            elif isinstance(result, dict):
                if result.get('label') == 'ENTAILMENT':
                    entailment_score = result.get('score', 0.0)

            # Threshold: 0.3
            is_hallucinated = entailment_score < 0.3

            return {
                "is_hallucinated": is_hallucinated,
                "entailment_score": float(entailment_score),
                "confidence": 1.0 - entailment_score if is_hallucinated else entailment_score,
                "threshold_used": 0.3,
                "label": result[0].get('label') if isinstance(result, list) and len(result) > 0 else "UNKNOWN"
            }

        except Exception as e:
            print(f"⚠️ NLI error: {e}")
            return {
                "is_hallucinated": False,
                "entailment_score": 0.5,
                "confidence": 0.5,
                "reason": f"Error: {str(e)[:100]}"
            }

    def validate_answer(self, answer: str, query: str, context: str, confidence: float = 0.5) -> Dict:
        """Main validation function"""
        if confidence > 0.9:
            return {
                "validation_skipped": True,
                "reason": "High confidence threshold met",
                "final_validation": "PASS",
                "confidence": float(confidence)
            }

        correctness_score = self.heuristic_correctness(answer, query)
        hallucination_result = self.hallucination_check(answer, context)

        # Looser correctness threshold: 0.4 → 0.35
        passes_correctness = correctness_score > 0.35
        passes_hallucination = not hallucination_result.get("is_hallucinated", False)

        final_validation = "PASS" if (passes_correctness and passes_hallucination) else "FAIL"

        return {
            "validation_skipped": False,
            "correctness_score": correctness_score,
            "hallucination_check": hallucination_result,
            "passes_correctness": passes_correctness,
            "passes_hallucination": passes_hallucination,
            "final_validation": final_validation,
            "confidence": float((correctness_score + hallucination_result.get("entailment_score", 0.5)) / 2)
        }

    def regenerate_answer(self, query: str, chunks: List[Dict], previous_answer: str,
                         answer_generator: 'AnswerGenerator', pdf_processor: PDFProcessor,
                         query_embedding: np.ndarray = None, max_attempts: int = 2) -> Dict:
        """Regenerate answer with more context if validation fails"""
        for attempt in range(max_attempts):
            print(f"  Regeneration attempt {attempt + 1}/{max_attempts}")

            expanded_chunks = chunks[:8 + (attempt * 4)]  # More context each time

            new_result = answer_generator.get_answer(
                query, expanded_chunks, pdf_processor, query_embedding, skip_cache=True
            )

            context = answer_generator.format_context(
                answer_generator.get_parent_chunks(expanded_chunks, pdf_processor)[:8]
            )

            validation = self.validate_answer(
                new_result["answer"], query, context, confidence=0.5
            )

            if validation["final_validation"] == "PASS":
                new_result["regeneration_attempts"] = attempt + 1
                new_result["validation"] = validation
                return new_result

        new_result["regeneration_attempts"] = max_attempts
        new_result["validation"] = validation
        new_result["answer"] = f"{new_result['answer']}\n\n⚠️ Note: This answer may not be fully validated."
        return new_result

if __name__ == "__main__":
    print("✅ Phase 5: Answer Validator FIXED")
    print("   - Looser NLI threshold (0.5 → 0.3)")
    print("   - Looser correctness threshold (0.4 → 0.35)")

In [ ]:
# Run this cell to fix the NLTK error
import nltk
nltk.download('stopwords')
nltk.download('punkt')

print("✅ NLTK data downloaded. Now continue with your workflow.")

In [ ]:
# ============================================
# PHASE 6: Learning Loop & RAG Orchestrator (Colab Version - FULLY FIXED)
# ============================================

import json
from datetime import datetime
from typing import Dict, List, Any
import os
import numpy as np

class LearningLoop:
    def __init__(self, log_file: str = "rag_logs.jsonl"):
        self.log_file = log_file
        self.feedback_data = []

    def log_interaction(self, query: str, rewritten_query: str, result: Dict, feedback: str = None):
        """Log query, rewritten query, result, and feedback"""
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "query": query,
            "rewritten_query": rewritten_query,
            "result": {
                "answer": result.get("answer", ""),
                "confidence": float(result.get("confidence", 0)),
                "chunks_used": result.get("chunks_used", 0),
                "parents_used": result.get("parents_used", 0),
                "cache_hit": result.get("cache_hit", False),
                "semantic_cache": result.get("semantic_cache", False),
                "query_type": result.get("query_type", "unknown"),
                "validation": result.get("validation", {}),
                "tokens_used": result.get("tokens_used", 0)
            },
            "feedback": feedback,
            "route": result.get("route", "unknown")
        }

        with open(self.log_file, 'a') as f:
            f.write(json.dumps(log_entry) + '\n')

        self.feedback_data.append(log_entry)
        print(f"📝 Logged: '{query[:30]}...' → '{rewritten_query[:30]}...'")

    def analyze_logs(self, days_back: int = 7):
        """Analyze logs with rewrite and cache stats"""
        if not os.path.exists(self.log_file):
            return {"error": "No logs found"}

        logs = []
        with open(self.log_file, 'r') as f:
            for line in f:
                logs.append(json.loads(line))

        total_queries = len(logs)
        cache_hits = sum(1 for log in logs if log["result"].get("cache_hit", False))
        semantic_hits = sum(1 for log in logs if log["result"].get("semantic_cache", False))

        avg_confidence = np.mean([log["result"].get("confidence", 0) for log in logs]) if logs else 0

        # Query rewrite analysis
        rewritten_count = sum(1 for log in logs if log["query"] != log["rewritten_query"])

        feedbacks = [log.get("feedback") for log in logs if log.get("feedback")]
        positive_feedback = sum(1 for fb in feedbacks if fb in ["👍", "positive", "good"])
        negative_feedback = sum(1 for fb in feedbacks if fb in ["👎", "negative", "bad"])

        return {
            "total_queries": total_queries,
            "cache_hit_rate": cache_hits / max(total_queries, 1),
            "semantic_cache_hit_rate": semantic_hits / max(total_queries, 1),
            "average_confidence": float(avg_confidence),
            "queries_rewritten": rewritten_count,
            "rewrite_rate": rewritten_count / max(total_queries, 1),
            "positive_feedback": positive_feedback,
            "negative_feedback": negative_feedback,
            "feedback_ratio": positive_feedback / max(positive_feedback + negative_feedback, 1),
            "recent_logs": logs[-10:]
        }

    def retrain_thresholds(self, router):
        """Adjust thresholds based on feedback"""
        analysis = self.analyze_logs()

        if analysis.get("negative_feedback", 0) > 5:
            negative_rate = analysis["negative_feedback"] / max(
                analysis["positive_feedback"] + analysis["negative_feedback"], 1
            )

            if negative_rate > 0.3:
                # Make system more conservative
                router.unrelated_threshold *= 0.95  # Slight decrease
                router.high_confidence_threshold *= 1.05  # Slight increase
                print(f"⚠️ Adjusted thresholds based on feedback")

        return router

class RAGOrchestrator:
    def __init__(self, openai_api_key: str = None, pdf_folder: str = None,
                 base_url: str = None, model: str = None):

        print("🚀 Initializing RAG System with ALL FIXES...")

        self.pdf_processor = PDFProcessor()
        self.query_processor = QueryProcessor(self.pdf_processor)
        self.router = SmartRouter()
        self.adaptive_processor = AdaptiveProcessor(self.pdf_processor)
        self.answer_generator = AnswerGenerator(
            api_key=openai_api_key,
            base_url=base_url,
            model=model
        )
        self.external_retriever = ExternalRetriever()
        # Pass embedding model to answer generator for semantic cache
        self.answer_generator.set_embedding_model(self.pdf_processor.embedding_model)

        self.validator = AnswerValidator()
        self.learning_loop = LearningLoop()
       # self.domain_keywords = []
        print("✅ RAG System initialized with ALL FIXES!")

    def upload_and_process_pdf(self, pdf_file) -> Dict:
        """Process a single uploaded PDF file with deduplication"""
        try:
            # Handle Gradio file object
            if hasattr(pdf_file, 'name'):
                # It's a file path
                file_path = pdf_file.name

                # File-level deduplication - read from path, not object
                with open(file_path, 'rb') as f:
                    file_hash = hashlib.md5(f.read()).hexdigest()

                pdf_name = os.path.basename(file_path)

                # Extract text using file path
                text = self.pdf_processor.extract_text(file_path)

            else:
                # Fallback for direct file objects
                file_content = pdf_file.read()
                file_hash = hashlib.md5(file_content).hexdigest()
                pdf_file.seek(0)
                pdf_name = getattr(pdf_file, 'name', 'uploaded.pdf')
                text = self.pdf_processor.extract_text(pdf_name)  # May fail

            # Check for duplicate
            if hasattr(self.pdf_processor, 'processed_file_hashes'):
                if file_hash in self.pdf_processor.processed_file_hashes:
                    return {
                        "success": False,
                        "error": "This PDF has already been uploaded",
                        "duplicate": True,
                        "chunks_added": 0,
                        "parents_added": 0
                    }

            # Process PDF with parent-child chunking
            # Create parent chunks
            parents = self.pdf_processor.create_parent_chunks(text, pdf_name)

            # Create child chunks from parents (with deduplication)
            children = self.pdf_processor.create_child_chunks(parents)

            if not children:
                return {
                    "success": False,
                    "error": "No new chunks added (all chunks were duplicates)",
                    "duplicate": True,
                    "chunks_added": 0,
                    "parents_added": 0
                }

            # Create embeddings for children
            embeddings = self.pdf_processor.create_embeddings(children)

            # Build or update FAISS index
            if self.pdf_processor.vector_db is None:
                self.pdf_processor.build_faiss_index(embeddings)
            else:
                self.pdf_processor.vector_db.add(embeddings.astype('float32'))

            # Store chunks
            self.pdf_processor.parent_chunks.extend(parents)
            self.pdf_processor.child_chunks.extend(children)
            self.pdf_processor.processed_file_hashes.add(file_hash)

            return {
                "success": True,
                "filename": pdf_name,
                "parents_added": len(parents),
                "chunks_added": len(children),
                "total_parents": len(self.pdf_processor.parent_chunks),
                "total_children": len(self.pdf_processor.child_chunks)
            }
        except Exception as e:
            return {
                "success": False,
                "error": str(e),
                "chunks_added": 0,
                "parents_added": 0
            }

    def clear_all_cache(self):
        """Clear all caches for fresh testing"""
        self.answer_generator.type_aware_cache = {}
        self.answer_generator.semantic_cache = None
        self.answer_generator.set_embedding_model(self.pdf_processor.embedding_model)
        # Remove the lines below if they cause errors
        # self.query_processor.semantic_cache = SemanticCache(...)
        # self.query_processor.exact_answer_cache = {}
        print("✅ Answer generator cache cleared")


    def process_query(self, query: str) -> Dict:
        """Main orchestrator with external fallback"""
        print(f"\n{'='*60}")
        print(f"📥 Query: {query}")
        print(f"{'='*60}")

        # PHASE 1: Query Processing
        print("\n🔷 PHASE 1: Query Processing")
        phase1_result = self.query_processor.process_query_parallel(query)
        rewritten_query = phase1_result.get("rewritten_query", query)

        # Check for cache hit
        if phase1_result.get("cache_hit"):
            print("  ✅ SEMANTIC CACHE HIT - returning cached answer")
            result = {
                "answer": phase1_result["cached_answer"],
                "confidence": phase1_result["max_similarity"],
                "route": "cache_hit",
                "chunks_used": 0,
                "cache_hit": True,
                "semantic_cache": True,
                "rewritten_query": rewritten_query,
                "timestamp": datetime.now().isoformat()
            }
            self.learning_loop.log_interaction(query, rewritten_query, result)
            return result

        query_embedding = self.query_processor.embed_query(rewritten_query)
        child_chunks = phase1_result.get("retrieved_chunks", [])

        if not child_chunks:
            phase1_result = self.query_processor.process_query_parallel(query)
            child_chunks = phase1_result.get("retrieved_chunks", [])

        # PHASE 2: Smart Routing (with external fallback decision)
        print("\n🔷 PHASE 2: Smart Routing")
        pdf_has_content = len(self.pdf_processor.child_chunks) > 0
        route_decision = self.router.route_query(
            rewritten_query,
            phase1_result["max_similarity"],
            pdf_has_content
        )
        print(f"  Route: {route_decision['route']}")
        print(f"  Action: {route_decision['action']}")
        print(f"  Confidence: {phase1_result['max_similarity']:.3f}")

        # Handle external_only route (no PDFs)
        if route_decision["route"] == "external_only":
            print("\n🔷 No documents - using external sources only")
            external_chunks = self.external_retriever.search(rewritten_query)

            if not external_chunks:
                result = {
                    "answer": "No documents uploaded and no external information found. Please upload PDFs or try a different question.",
                    "confidence": 0.0,
                    "route": "external_only",
                    "chunks_used": 0,
                    "rewritten_query": rewritten_query
                }
                self.learning_loop.log_interaction(query, rewritten_query, result)
                return result

            for chunk in external_chunks:
                chunk["is_external"] = True
                chunk["type"] = "external"

            answer_result = self.answer_generator.get_answer(
                rewritten_query, external_chunks, self.pdf_processor, query_embedding
            )
            answer_result["external_used"] = True

            result = {
                "answer": answer_result["answer"],
                "confidence": 0.5,
                "route": "external_only",
                "chunks_used": len(external_chunks),
                "external_used": True,
                "rewritten_query": rewritten_query,
                "timestamp": datetime.now().isoformat()
            }
            self.learning_loop.log_interaction(query, rewritten_query, result)
            return result

        # Early exit: Unrelated query
        if route_decision["route"] == "unrelated":
            answer = self.router.polite_refusal(query, phase1_result["max_similarity"])
            result = {
                "answer": answer,
                "confidence": float(route_decision["confidence"]),
                "route": route_decision["route"],
                "reason": route_decision["reason"],
                "chunks_used": 0,
                "rewritten_query": rewritten_query
            }
            self.learning_loop.log_interaction(query, rewritten_query, result)
            return result

        # External fallback route with topic filtering
        external_chunks = []
        if route_decision["route"] == "external_fallback":
            print("\n🔷 EXTERNAL FALLBACK: Document content insufficient")

            # Use cross-encoder for relevance (more accurate than cosine similarity)
            is_relevant = True
            if self.adaptive_processor.sufficiency_model and self.pdf_processor.child_chunks:
                try:
                    # Take top 5 chunks as representatives
                    sample_chunks = self.pdf_processor.child_chunks[:5]
                    chunk_texts = [chunk.get("text", "")[:500] for chunk in sample_chunks]
                    pairs = [(rewritten_query, text) for text in chunk_texts]
                    scores = self.adaptive_processor.sufficiency_model.predict(pairs)
                    avg_score = float(np.mean(scores))
                    print(f"  Cross-encoder relevance score: {avg_score:.2f}")
                    is_relevant = avg_score > 2.0  # Threshold for relevance
                except Exception as e:
                    print(f"  ⚠️ Cross-encoder error: {e}, falling back to similarity")
                    is_relevant = phase1_result["max_similarity"] > 0.75

            if not is_relevant:
                print(f"  ⚠️ Query off-topic - refusing")
                result = {
                    "answer": "I can only answer questions related to the content of your uploaded documents. Your documents cover Natural Language Processing (NLP).",
                    "confidence": 0.0,
                    "route": "unrelated",
                    "reason": "Query not relevant to document content (cross-encoder check)",
                    "chunks_used": 0,
                    "rewritten_query": rewritten_query,
                    "external_used": False,
                    "timestamp": datetime.now().isoformat()
                }
                self.learning_loop.log_interaction(query, rewritten_query, result)
                return result

            # Query is relevant - proceed with external search
            print(f"  📚 Query relevant - using external search")
            external_chunks = self.external_retriever.search(rewritten_query)

            if external_chunks:
                print(f"  Found {len(external_chunks)} external sources")
                for chunk in external_chunks:
                    chunk["is_external"] = True
                    chunk["type"] = "external"
            else:
                print("  No external sources found - using documents only")
        if route_decision["route"] == "needs_processing" and not external_chunks:
            print("\n🔷 CHECKING FOR EXTERNAL SEARCH (needs_processing route)")
            if phase1_result["max_similarity"] < 0.70:  # Medium confidence
                print(f"  📚 Medium confidence ({phase1_result['max_similarity']:.3f}) - trying external search")
                external_chunks = self.external_retriever.search(rewritten_query)
                if external_chunks:
                    print(f"  Found {len(external_chunks)} external sources")
                    for chunk in external_chunks:
                        chunk["is_external"] = True
                        chunk["type"] = "external"
                else:
                    print("  No external sources found")

        # PHASE 3: Adaptive Processing
        retrieval_attempts = 1
        if route_decision["route"] in ["needs_processing", "specific_query"]:
            print("\n🔷 PHASE 3: Adaptive Processing")
            child_chunks, retrieval_attempts = self.adaptive_processor.hybrid_search(
                rewritten_query, child_chunks, max_loops=2
            )
            print(f"  Final child chunks: {len(child_chunks)} (attempts: {retrieval_attempts})")

        # Merge document chunks with external chunks
        all_chunks = child_chunks.copy()
        has_external = len(external_chunks) > 0
        if has_external:
            all_chunks.extend(external_chunks)
            print(f"  Total chunks (documents + external): {len(all_chunks)}")

        # PHASE 4: Answer Generation
        print("\n🔷 PHASE 4: Answer Generation")

        self.answer_generator.has_external = has_external

        answer_result = self.answer_generator.get_answer(
            rewritten_query,
            all_chunks,
            self.pdf_processor,
            query_embedding,
            skip_cache=True
        )
        answer_result["external_used"] = has_external

        # PHASE 5: Smart Validation (Skip for cached answers)
        print("\n🔷 PHASE 5: Smart Validation")

        if answer_result.get("cache_hit", False):
            print("  ⚠️ Answer from cache - skipping validation")
            validation = {
                "final_validation": "PASS",
                "validation_skipped": True,
                "reason": "Answer from cache",
                "correctness_score": 1.0,
                "passes_correctness": True,
                "passes_hallucination": True
            }
        else:
            # For high confidence answers, skip validation entirely
            if route_decision["confidence"] > 0.70:
                print(f"  ✅ High confidence answer ({route_decision['confidence']:.3f}) - skipping validation")
                validation = {
                    "final_validation": "PASS",
                    "validation_skipped": True,
                    "reason": "High confidence"
                }
            else:
                parent_chunks = self.answer_generator.get_parent_chunks(
                    child_chunks, self.pdf_processor
                )

                validation_context = self.answer_generator.format_context(parent_chunks, max_chunks=5)
                if has_external:
                    external_text = "\n".join([chunk["text"][:500] for chunk in external_chunks[:2]])
                    validation_context += f"\n\nExternal Sources:\n{external_text}"

                validation = self.validator.validate_answer(
                    answer_result["answer"], rewritten_query, validation_context,
                    confidence=route_decision["confidence"]
                )

                if validation["final_validation"] == "FAIL" and not validation.get("validation_skipped", False):
                    print("  Validation FAILED - regenerating...")
                    answer_result = self.validator.regenerate_answer(
                        rewritten_query, all_chunks, answer_result["answer"],
                        self.answer_generator, self.pdf_processor,
                        query_embedding, max_attempts=2
                    )
                    validation = answer_result.get("validation", validation)

        if validation["final_validation"] == "PASS" and not answer_result.get("cache_hit", False):
            print("  ✅ Validation passed - caching answer")
            if self.answer_generator.semantic_cache and query_embedding is not None:
                query_type = answer_result.get("query_type", "general")
                self.answer_generator.semantic_cache.add(
                    rewritten_query,
                    query_embedding,
                    answer_result["answer"],
                    metadata={"query_type": query_type}
                )
                cache_key = hashlib.md5(
                    query_embedding.tobytes() + query_type.encode()
                ).hexdigest()
                cached_result = answer_result.copy()
                cached_result["cache_hit"] = True
                cached_result["semantic_cache"] = True
                self.answer_generator.type_aware_cache[cache_key] = cached_result
                print("  ✅ Answer cached after validation")
        if has_external and not answer_result.get("cache_hit", False):
            print("  ✅ External answer - caching anyway")
            if self.answer_generator.semantic_cache and query_embedding is not None:
                query_type = answer_result.get("query_type", "general")
                self.answer_generator.semantic_cache.add(
                    rewritten_query,
                    query_embedding,
                    answer_result["answer"],
                    metadata={"query_type": query_type}
                )
                cache_key = hashlib.md5(
                    query_embedding.tobytes() + query_type.encode()
                ).hexdigest()
                cached_result = answer_result.copy()
                cached_result["cache_hit"] = True
                cached_result["semantic_cache"] = True
                self.answer_generator.type_aware_cache[cache_key] = cached_result
                print("  ✅ External answer cached")
        # Prepare final result
        result = {
            "answer": answer_result["answer"],
            "confidence": float(route_decision["confidence"]),
            "route": route_decision["route"],
            "chunks_used": answer_result.get("chunks_used", 0),
            "parents_used": answer_result.get("parents_used", 0),
            "retrieval_attempts": retrieval_attempts,
            "cache_hit": answer_result.get("cache_hit", False),
            "semantic_cache": answer_result.get("semantic_cache", False),
            "query_type": answer_result.get("query_type", "general"),
            "validation": validation,
            "tokens_used": int(answer_result.get("tokens", 0)),
            "rewritten_query": rewritten_query,
            "external_used": has_external,
            "timestamp": datetime.now().isoformat()
        }

        # PHASE 6: Learning Loop
        print("\n🔷 PHASE 6: Learning Loop")
        self.learning_loop.log_interaction(query, rewritten_query, result)

        if len(self.learning_loop.feedback_data) % 10 == 0:
            self.router = self.learning_loop.retrain_thresholds(self.router)

        print(f"\n✅ Processing complete!")
        print(f"   Confidence: {result['confidence']:.3f}")
        print(f"   Route: {result['route']}")
        print(f"   External used: {has_external}")
        print(f"   Tokens: {result['tokens_used']}")
        print(f"{'='*60}")

        return result

# ============================================
# MAIN EXECUTION CELL
# ============================================

if __name__ == "__main__":
    print("=" * 60)
    print("🚀 Colab RAG System - COMPLETE FIXED VERSION")
    print("=" * 60)

    try:
        from google.colab import userdata
        print("✅ Colab environment detected")
    except:
        print("⚠️ Not in Colab - using environment variables")

    import os

    try:
        os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
        os.environ["OPENAI_BASE_URL"] = userdata.get('OPENAI_BASE_URL', "https://apidev.navigatelabsai.com/")
        os.environ["OPENAI_MODEL"] = userdata.get('OPENAI_MODEL', "gpt-4.1-nano")
        print("✅ Loaded API details from Colab Secrets")
    except:
        print("⚠️ Set environment variables manually")
        os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
        os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL", "https://apidev.navigatelabsai.com/")
        os.environ["OPENAI_MODEL"] = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")

    # Initialize the system
    rag_system = RAGOrchestrator(
        openai_api_key=os.getenv("OPENAI_API_KEY"),
        pdf_folder=None,
        base_url=os.getenv("OPENAI_BASE_URL"),
        model=os.getenv("OPENAI_MODEL")
    )

    print("\n" + "=" * 60)
    print("✅ SYSTEM READY - ALL FIXES APPLIED")
    print("=" * 60)
    print("📌 FIXES INCLUDED:")
    print("   • Parent-child chunking")
    print("   • File & chunk deduplication")
    print("   • Query rewriter (typos + normalization)")
    print("   • Embedding-based semantic cache")
    print("   • Relevance check before expansion")
    print("   • Parent retrieval (child → parent)")
    print("   • MMR diversity reranking")
    print("   • Structured output prompts")
    print("   • Temperature = 0 (deterministic)")
    print("   • Looser validation thresholds")
    print("   • Multi-PDF ready")
    print("=" * 60)

In [ ]:
# ============================================
# GRADIO UI FOR COLAB - COMPLETE CLEAN VERSION
# ============================================

import gradio as gr
import os
import numpy as np

try:
    from google.colab import userdata
    IN_COLAB = True
    print("✅ Running in Google Colab")
except:
    IN_COLAB = False
    print("⚠️ Running locally")

if IN_COLAB:
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
        os.environ["OPENAI_BASE_URL"] = userdata.get('OPENAI_BASE_URL', "https://apidev.navigatelabsai.com/")
        os.environ["OPENAI_MODEL"] = userdata.get('OPENAI_MODEL', "gpt-4.1-nano")
        print("✅ Loaded from Colab Secrets")
    except Exception as e:
        print(f"⚠️ Colab Secrets error: {e}")

# Initialize RAG system
rag_system = RAGOrchestrator(
    pdf_folder=None,
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL", "https://apidev.navigatelabsai.com/"),
    model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano")
)

def process_pdfs(files):
    """Process multiple PDF files at once"""
    if not files:
        return "📭 No files selected"

    results = []
    for file in files:
        result = rag_system.upload_and_process_pdf(file)

        if result.get("success"):
            chunks = result.get('chunks_added', 0)
            parents = result.get('parents_added', 0)
            results.append(f"✅ {file.name}: {chunks} chunks, {parents} sections")

        elif result.get("duplicate"):
            results.append(f"⏭️ {file.name}: Skipped (duplicate)")

        else:
            error_msg = result.get('error', 'Failed')
            results.append(f"❌ {file.name}: {error_msg}")

    return "\n".join(results)

def ask_question(query, chat_history):
    """Process user question and return answer with metadata"""
    if not query.strip() or not rag_system:
        return chat_history

    try:
        result = rag_system.process_query(query)
        answer = result.get("answer", "No answer generated")
        confidence = result.get("confidence", 0)
        cache_hit = result.get("cache_hit", False)
        semantic_cache = result.get("semantic_cache", False)
        external_used = result.get("external_used", False)
        parents_used = result.get("parents_used", 0)

        # Format answer with metadata
        formatted_answer = f"{answer}\n\n"

        # External source indicator
        if external_used:
            formatted_answer += f"🌐 *Includes information from Wikipedia/arXiv*\n\n"

        # Confidence score with cache indicator
        if cache_hit:
            if semantic_cache:
                formatted_answer += f"📊 Confidence: {confidence:.2f} ⚡ (semantic cache)"
            else:
                formatted_answer += f"📊 Confidence: {confidence:.2f} ⚡ (exact cache)"
        else:
            formatted_answer += f"📊 Confidence: {confidence:.2f}"

        # Sections used (only if documents were used)
        if parents_used > 0:
            formatted_answer += f"\n📄 From {parents_used} document sections"

        chat_history.append([query, formatted_answer])

    except Exception as e:
        chat_history.append([query, f"❌ Error: {str(e)[:200]}"])

    return chat_history

def clear_chat():
    """Clear the chat history"""
    return []

# ============================================
# BUILD GRADIO UI
# ============================================

with gr.Blocks(theme=gr.themes.Soft(), title="Academic RAG Assistant") as demo:
    gr.Markdown("""
    # 📚 Academic RAG Assistant
    ### Upload PDFs and ask intelligent questions about their content
    """)

    with gr.Row():
        # LEFT COLUMN - PDF Upload
        with gr.Column(scale=1):
            gr.Markdown("### 📄 Upload Documents")
            pdf_input = gr.File(
                label="Upload PDF(s)",
                file_types=[".pdf"],
                file_count="multiple"
            )
            pdf_status = gr.Textbox(
                label="Status",
                interactive=False,
                lines=5,
                placeholder="Upload PDFs to get started..."
            )
            upload_btn = gr.Button("📤 Upload & Process", variant="primary", size="lg")

        # RIGHT COLUMN - Chat Interface
        with gr.Column(scale=2):
            gr.Markdown("### 💬 Ask Questions")
            chatbot = gr.Chatbot(
                label="Conversation",
                height=450,
                bubble_full_width=False
            )
            query_input = gr.Textbox(
                label="Your question",
                placeholder="Ask about the PDF content...",
                lines=2
            )

            with gr.Row():
                ask_btn = gr.Button("🚀 Ask", variant="primary", size="lg")
                clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary")

    # ============================================
    # EVENT HANDLERS
    # ============================================

    upload_btn.click(
        fn=process_pdfs,
        inputs=[pdf_input],
        outputs=[pdf_status]
    )

    ask_btn.click(
        fn=ask_question,
        inputs=[query_input, chatbot],
        outputs=[chatbot]
    ).then(
        fn=lambda: "",
        outputs=[query_input]
    )

    query_input.submit(
        fn=ask_question,
        inputs=[query_input, chatbot],
        outputs=[chatbot]
    ).then(
        fn=lambda: "",
        outputs=[query_input]
    )

    clear_btn.click(
        fn=clear_chat,
        outputs=[chatbot]
    )

    # ============================================
    # FOOTER WITH SYSTEM INFO
    # ============================================

    gr.Markdown("""
    ---
    ### 📌 System Features
    - **Document Q&A** - Answers from uploaded PDFs
    - **External Fallback** - Wikipedia + arXiv when documents are insufficient
    - **Smart Caching** - Similar questions answered instantly ⚡
    - **Hallucination Detection** - NLI model validates every answer
    - **Confidence Scores** - See how certain the system is
    """)

# ============================================
# LAUNCH THE APP
# ============================================

print("=" * 60)
print("🚀 Launching Academic RAG Assistant...")
print("=" * 60)
print("📊 API Status:", "✅ Ready" if os.getenv("OPENAI_API_KEY") else "❌ No API Key")
print("✅ Multi-PDF upload enabled")
print("✅ External fallback: Wikipedia + arXiv")
print("✅ Semantic cache active")
print("=" * 60)

demo.launch(share=True, debug=False)